# 01. Baseline Inference: Chest X-Ray Classification
## Зона ответственности: Напарник

**Цель этапа:**
1. Загрузка предобученной модели (ResNet / TorchXRayVision).
2. Прогон инференса по набору рентгенограмм органов грудной клетки (CXR).
3. Извлечение вероятностей и сырых логитов.
4. Сохранение предсказаний в `outputs/predictions.csv` для последующей конформной калибровки (MAPIE).

In [ ]:
# 1. Импорт библиотек
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

# Попытка импорта torchxrayvision при наличии
try:
    import torchxrayvision as xrv
    print("TorchXRayVision успешно импортирован!")
except ImportError:
    print("TorchXRayVision не найден, используется стандартный torchvision ResNet.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

### 2. Настройка путей и параметров

In [ ]:
DATA_DIR = Path("../data")
OUTPUTS_DIR = Path("../outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_CLASSES = 5  # Например: Normal, Atelectasis, Cardiomegaly, Effusion, Pneumonia
CLASS_NAMES = ["Normal", "Atelectasis", "Cardiomegaly", "Effusion", "Pneumonia"]

print(f"Директория данных: {DATA_DIR.resolve()}")
print(f"Директория результатов: {OUTPUTS_DIR.resolve()}")

### 3. Загрузка предобученной модели ResNet

In [ ]:
def get_model(num_classes: int = len(CLASS_NAMES)):
    """Инициализация ResNet50 с заменой классификационной головы."""
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = torch.nn.Linear(in_features, num_classes)
    model.eval()
    return model.to(device)

model = get_model()
print(model)

### 4. Пайплайн предобработки изображений (Transforms)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

### 5. Инференс и вычисление логитов / вероятностей

In [ ]:
@torch.no_grad()
def run_inference(model, dataloader):
    """Прогон инференса с извлечением logits и softmax/sigmoid probabilities."""
    all_logits = []
    all_probs = []
    all_targets = []
    
    for images, targets in tqdm(dataloader, desc="Inference"):
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=-1)
        
        all_logits.append(logits.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_targets.append(targets.numpy())
        
    return (
        np.vstack(all_logits),
        np.vstack(all_probs),
        np.concatenate(all_targets)
    )

print("Пайплайн инференса подготовлен к обработке тестового датасета.")

### 6. Сохранение результатов в `outputs/predictions.csv`

In [ ]:
# Пример структуры сохранения таблицы предсказаний
# df_out = pd.DataFrame(all_probs, columns=[f"prob_{name}" for name in CLASS_NAMES])
# df_out['true_label'] = all_targets
# df_out.to_csv(OUTPUTS_DIR / 'predictions.csv', index=False)
print("Таблица predictions.csv будет сохранена в папку outputs/")